<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Notes/11-supervised_regression/11-supervised_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 11: Supervised Learning II, Regression

**DSC 210 Foundations of Data Science**

References:
- [Hands-on Introduction to Data Science with Python](https://florian-huber.github.io/data_science_course/) (CC BY-NC-SA 4.0)
- [scikit-learn user guide](https://scikit-learn.org/stable/supervised_learning.html), on linear models, trees, and cross-validation
- Auto MPG (Quinlan, 1993) and the Titanic passenger list, both included with `seaborn`

```
ASK  ->  GET  ->  EXPLORE  ->  [ MODEL ]  ->  COMMUNICATE
```

*Last major revision: 2026-08-13*

Module 10 predicted a **category**. Today the label is a **number**, and almost everything changes with it: the models, the error measures, and what counts as a good answer.

## Key Concepts

- Interpret a **slope** and **intercept** in the units of the problem, and recognize unsafe extrapolation
- Evaluate a regression with **MAE**, **RMSE**, and $R^2$, and read a **residual plot**
- Use **k-NN**, **decision tree**, and **random forest** regression, and explain the shape of fit each produces
- Explain why a straight line fails on a 0/1 label, and use **logistic regression** to predict a probability
- Use **k-fold cross-validation**, and explain why a single train/test split can mislead

---
## 1. When the Label Is a Number
---

Module 10 asked *which species is this penguin?* Today we ask *how many miles per gallon does this car get?*

**Definition.** In **regression** the label is numeric, and a prediction can be close without being right. In **classification** the label is a category, and a prediction is either correct or not.

| | Classification (Module 10) | Regression (today) |
| --- | --- | --- |
| Label | a category | a number |
| A prediction is | right or wrong | off by some amount |
| Typical error measure | accuracy, precision, recall | MAE, RMSE, $R^2$ |
| Confusion matrix? | yes | no; there are no classes to confuse |
| Baseline to beat | always guess the common class | always guess the mean |

Hold on to that last row. It is the regression version of the fraud-detection lesson: **never report an error without saying what doing nothing would have scored.**

In [ ]:
# RUN-TOGETHER
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# A new dataset: 392 car models built between 1970 and 1982.
cars = sns.load_dataset('mpg').dropna()
cars['weight_1000lb'] = cars['weight'] / 1000     # nicer units for a slope

print(cars.shape)
cars[['name', 'model_year', 'weight_1000lb', 'mpg']].head(3)

---
## 2. Linear Regression
---
### 2.1 The line, and what it minimizes

A linear model predicts the label as a straight-line function of a feature:

$$\hat{y} = b_0 + b_1 x$$

For any candidate line, each observation has a **residual**, the vertical gap between what happened and what the line predicted:

$$e_i = y_i - \hat{y}_i$$

**Definition.** The **least squares** line is the one that minimizes the **sum of squared errors**, $\text{SSE} = \sum (y_i - \hat{y}_i)^2$.

Squaring does two jobs: overshoots and undershoots both count as error, and one large miss is penalized more than several small ones.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/11-supervised_regression/fig_least_squares_residuals.png?raw=true" width="640">

*Each red segment is one residual. Least squares slides and tilts the line until the total squared length of those segments is as small as it can be. There is a formula for the answer, and `scikit-learn` will apply it for us.*

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/11-supervised_regression/fig_linear_regression_fitting.png?raw=true" width="620">

### 2.2 Fitting the line

**How much fuel efficiency does a car lose as it gets heavier?** Weight is measured in thousands of pounds so the slope reads in convenient units.

In [ ]:
# RUN-TOGETHER
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = cars[['weight_1000lb']]
y = cars['mpg']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=210)

lin = LinearRegression().fit(X_train, y_train)
pred = lin.predict(X_test)

print('slope    :', round(lin.coef_[0], 2), 'mpg per 1000 lb')
print('intercept:', round(lin.intercept_, 2), 'mpg')
print('observed weight range:', round(cars['weight_1000lb'].min(), 2),
      'to', round(cars['weight_1000lb'].max(), 2), 'thousand lb')

#### **Activity 11.1 - Reading a fitted line**

You now have a fitted model. The skill is not producing it; it is saying what it means and knowing where it stops being trustworthy.

**A. Say the slope in a sentence.** Fill in: *"Each additional 1,000 pounds is associated with a change of ______ mpg."* Is that an increase or a decrease, and does the sign make physical sense?

**B. Make three predictions by hand from $\hat{y} = b_0 + b_1 x$.**

| car | weight (1000 lb) | predicted mpg |
| --- | --- | --- |
| a light 1970s compact | 1.8 |  |
| a mid-size sedan | 3.0 |  |
| a full-size wagon | 4.8 |  |

**C. The intercept.** The model reports an intercept of about 46 mpg. In plain words, what car does that describe? Why is the number not usable, and what is the name for making a prediction outside the range the model was fitted on?

**D. A residual.** A real 1970 Plymouth in this dataset weighs 3.436 thousand pounds and gets 18 mpg. Compute its predicted value and its residual. Did the model over- or under-predict this car?

**E. Two students disagree.** One says "the slope proves that adding weight to a car causes worse fuel economy." The other says "all we have shown is that heavy cars in this sample tended to get worse mileage." Who is right, and which module made this distinction? Name one *other* difference between heavy and light 1970s cars that could be doing the work.

#### **Class Example 11.1 - Where the line fails**

Linear regression has three characteristic failures, and they are worth naming before we trust any line.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/11-supervised_regression/fig_linear_regression_issues.png?raw=true" width="900">

**The three standing limitations of linear regression:**

1. **It assumes a straight line.** If the truth is curved, the line will be systematically wrong in a pattern (panel 1). Section 3's residual plot is how you detect this.
2. **It makes impossible predictions.** A line is unbounded, so it will happily predict a negative price or a probability of 1.4 (panel 2). This is what motivates logistic regression in Section 6.
3. **It is over-confident far from the data.** The line extends forever and reports no doubt at all about a region where it has seen nothing (panel 3). This is Activity 11.1 part C.

---
## 3. Evaluating a Regression
---

A classifier is right or wrong. A regression is off by some amount, so we summarize the misses.

**Definition.** For predictions $\hat{y}_i$ against actuals $y_i$:

$$\text{MAE} = \frac{1}{n}\sum \lvert y_i - \hat{y}_i \rvert \qquad
\text{RMSE} = \sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2} \qquad
R^2 = 1 - \frac{\sum (y_i-\hat{y}_i)^2}{\sum (y_i-\bar{y})^2}$$

- **MAE** is the average miss, in the original units. Easiest to explain to a non-specialist.
- **RMSE** is also in the original units but squares first, so a few large misses dominate it. RMSE is always at least MAE, and the gap between them tells you how uneven the errors are.
- $R^2$ compares your model against the **do-nothing baseline of always predicting $\bar{y}$**. Zero means you matched that baseline, one means perfect, and it **can be negative**, which means you did worse than guessing the mean.

In [ ]:
# RUN-TOGETHER
print('MAE :', round(mean_absolute_error(y_test, pred), 2), 'mpg')
print('RMSE:', round(mean_squared_error(y_test, pred) ** 0.5, 2), 'mpg')
print('R2  :', round(r2_score(y_test, pred), 3))
print()
baseline = np.full_like(y_test, y_train.mean(), dtype=float)
print('baseline MAE (always predict the training mean):',
      round(mean_absolute_error(y_test, baseline), 2), 'mpg')

An average miss of about **3.3 mpg**, against a baseline of about **6.9 mpg**. The model roughly halves the error, and $R^2 = 0.700$ says the same thing on a different scale.

**The residual plot is the diagnostic no summary number can replace.** Plot the residuals against the predictions and look for shape.

In [ ]:
# RUN-TOGETHER
resid = y_test - pred

sns.scatterplot(x=pred, y=resid, alpha=0.7)
plt.axhline(0, color='crimson', linestyle='--')
plt.xlabel('predicted mpg'); plt.ylabel('residual (actual - predicted, mpg)')
plt.title('Residual plot for the weight model')
plt.show()

Three patterns are worth being able to recognize on sight.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/11-supervised_regression/fig_residual_patterns.png?raw=true" width="960">

*Only the first is what you hope for. The second says the model has the wrong shape and a line cannot fix it. The third says the errors grow with the prediction, so a single MAE hides how bad the large predictions are.*

> **Discuss.** Our car residuals are not quite shapeless: grouped into thirds by predicted mpg, the average residual is about $+0.26$ for the lightest third, $+0.07$ in the middle, and $+0.40$ for the heaviest. Which of the three panels does that most resemble, and what does it suggest about the true relationship between weight and mpg? Note that $R^2 = 0.700$ says nothing about this at all.

---
## 4. k-NN Regression
---

The algorithm from Module 10 needs almost no change to predict a number. Find the $k$ nearest neighbours as before, and instead of taking a **majority vote**, take their **average**.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/11-supervised_regression/fig_knn_algorithm.png?raw=true" width="760">

*Left, classification: the three neighbours vote. Right, regression: their values are averaged, giving 43.33. Same neighbours, same distance, different last step.*

Everything from Module 10 carries over, including the parts that bite: features must be **scaled**, and `k` must be chosen.

In [ ]:
# RUN-TOGETHER
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
Xtr_s, Xte_s = scaler.transform(X_train), scaler.transform(X_test)

print(f"{'k':>4}  {'MAE (mpg)':>10}  {'R2':>7}")
for k in (1, 5, 25):
    knn = KNeighborsRegressor(n_neighbors=k).fit(Xtr_s, y_train)
    q = knn.predict(Xte_s)
    print(f'{k:>4}  {mean_absolute_error(y_test, q):>10.2f}  {r2_score(y_test, q):>7.3f}')
print(f"{'line':>4}  {mean_absolute_error(y_test, pred):>10.2f}  {r2_score(y_test, pred):>7.3f}")

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/11-supervised_regression/fig_knn_regression.png?raw=true" width="960">

*Because every prediction is an average of nearby points, the fitted curve is a **staircase**, not a line. At k = 1 each step is one car. At k = 40 the steps are wide and the curve is smooth, but it flattens at the edges where neighbours run out.*

**`k = 1` is much the worst** ($R^2 = 0.390$ against the line's 0.700): the prediction is one other car's mpg, noise and all. **At `k = 25` the two are essentially tied.** So why prefer the line? Because it hands you *"7.7 mpg per 1000 pounds"*, a sentence you can put in a report. k-NN hands you predictions and no explanation.

---
## 5. Tree-Based Regression
---
### 5.1 A regression tree

Module 10's decision tree asked yes/no questions and put a **category** in each leaf. For regression the tree is identical except that each leaf holds a **number**: the mean of the training cases that land there. Instead of maximizing Gini gain, the split is chosen to reduce squared error.

The consequence is visible immediately.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/11-supervised_regression/fig_tree_regression.png?raw=true" width="960">

*A tree predicts a constant inside each region, so the fit is a **step function**. With `max_depth = 1` there are two steps. Unlimited, there is a step for almost every car, which is overfitting made visual.*

In [ ]:
# RUN-TOGETHER
from sklearn.tree import DecisionTreeRegressor

for d in (1, 3, None):
    t = DecisionTreeRegressor(max_depth=d, random_state=0).fit(X_train, y_train)
    print(f'max_depth={str(d):9}  train R2={r2_score(y_train, t.predict(X_train)):.3f}   '
          f'test R2={r2_score(y_test, t.predict(X_test)):.3f}')

The unlimited tree reaches a training $R^2$ of 0.975 and a test $R^2$ of 0.377. That is the same train-test gap you saw in Module 10, Section 4.4, and it is worse here because a single feature gives the tree nothing but noise to memorize.

### 5.2 Random forests: many trees, averaged

A single deep tree is unstable. Change a few training cars and the splits move. The fix is an **ensemble**.

**Definition.** A **random forest** grows many trees, each on a different random resample of the data and each allowed to consider only a random subset of features at every split, then **averages** their predictions. The individual trees are deliberately noisy and disagree; averaging cancels much of that noise.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/11-supervised_regression/fig_random_forest.png?raw=true" width="920">

*Left: eight trees, each grown on a different resample, each jagged and none trustworthy alone. Right: the average of 200 such trees. The wobble is largely gone.*

#### **Activity 11.2 - When does a forest beat a line?**

So far every model has had exactly **one** feature. That is unusual, and it disadvantages trees, which earn their keep by combining features and bending.

Run the cell below. It fits four models twice: once with weight alone, and once with six features.

**Questions.**

**A.** With one feature, rank the four models by test $R^2$. Which wins, and by how much?

**B.** With six features, rank them again. What changed?

**C.** Look at the **train** column for the unlimited tree in the six-feature run. It is 1.000. Explain what the tree has done, and why its test score is nevertheless respectable this time.

**D.** The random forest is built from trees like that one. Explain in one sentence how averaging many overfit trees produces a model that is not overfit.

**E.** A colleague concludes "random forests are better than linear regression." Using both runs, write a one-sentence correction that says *when* each is preferable.

In [ ]:
# FILL-IN  (Activity 11.2)
from sklearn.ensemble import RandomForestRegressor

FEATURES = ['weight_1000lb', 'horsepower', 'displacement',
            'model_year', 'cylinders', 'acceleration']

for label, cols in [('one feature (weight)', ['weight_1000lb']),
                    ('six features', ____)]:
    Xa, Xb, ya, yb = train_test_split(cars[cols], cars['mpg'], test_size=0.25, random_state=210)
    print(f'--- {label} ---')
    for name, model in [('linear', LinearRegression()),
                        ('tree, depth 3', DecisionTreeRegressor(max_depth=3, random_state=0)),
                        ('tree, unlimited', DecisionTreeRegressor(random_state=0)),
                        ('random forest', RandomForestRegressor(n_estimators=200, random_state=0))]:
        model.fit(Xa, ya)
        print(f'  {name:17} train R2={r2_score(ya, model.predict(Xa)):.3f}   '
              f'test R2={r2_score(yb, model.predict(Xb)):.3f}')
    print()

### 5.3 Getting interpretation back

Averaging 200 trees destroys the readable rule that made a single tree attractive. Forests give some of it back through **feature importance**: how much each feature contributed to reducing error across the whole ensemble.

In [ ]:
# RUN-TOGETHER
from sklearn.ensemble import RandomForestRegressor
FEATURES = ['weight_1000lb', 'horsepower', 'displacement',
            'model_year', 'cylinders', 'acceleration']

rf = RandomForestRegressor(n_estimators=200, random_state=0).fit(cars[FEATURES], cars['mpg'])

importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
sns.barplot(x=importance.values, y=importance.index, orient='h')
plt.xlabel('feature importance'); plt.ylabel('')
plt.title('What the forest leaned on')
plt.show()
print(importance.round(3).sort_values(ascending=False))

> **Discuss.** `displacement`, `weight`, and `cylinders` take the top three places, and all three measure roughly the same thing: how big the engine and car are. What does that tell you about reading feature importances when the features are correlated? Compare with the linear model, where you could read a single slope in mpg per 1000 lb.

---
## 6. Logistic Regression
---
### 6.1 Why a line fails on a 0/1 label

Suppose the label is binary, coded 0 and 1, and we fit a straight line anyway.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/11-supervised_regression/fig_linear_vs_logistic.png?raw=true" width="900">

*Left: the line climbs past 1, so its output cannot be read as a probability, and it would keep climbing forever. Right: the logistic curve is trapped between 0 and 1 by construction.*

**Definition.** The **logistic** (or **sigmoid**) function is

$$\sigma(z) = \frac{1}{1 + e^{-z}} \qquad \text{where} \qquad z = b_0 + b_1 x_1 + \cdots + b_n x_n$$

It maps any real number into $(0,1)$, so the output reads as a **probability**. Predictions become categories by comparing to a threshold, usually 0.5, and since

$$\sigma(z) = 0.5 \iff z = 0$$

**the decision boundary sits exactly where $z = 0$.**

#### **Activity 11.3 - Logistic regression by hand**

A model predicts whether a student passes an exam from hours studied, with fitted coefficients

$$b_0 = -4, \qquad b_1 = 0.8, \qquad z = -4 + 0.8x$$

**Step 1. Complete the table.** *(You will need $e^{-1.6} \approx 0.2019$ and $e^{1.6} \approx 4.953$.)*

| hours $x$ | $z = -4 + 0.8x$ | $e^{-z}$ | $p = \dfrac{1}{1+e^{-z}}$ | predicted at threshold 0.5 |
| --- | --- | --- | --- | --- |
| 3 |  |  |  |  |
| 5 |  |  |  |  |
| 7 |  |  |  |  |

**Step 2. The boundary.** Solve $-4 + 0.8x = 0$ for $x$: ______ hours.

**Questions.**

**A.** What is $p$ at exactly the boundary? Explain why that had to be the answer, from the definition of $\sigma$.

**B.** Students at 3 and 7 hours sit the same distance from the boundary. Compare their probabilities. What symmetry of $\sigma$ does that reveal?

**C.** Does this model ever predict a probability of exactly 0 or exactly 1? Justify from the formula, and connect your answer to limitation 2 of linear regression.

**D.** The department wants to flag at-risk students and would rather over-flag than miss someone. Should the threshold move above or below 0.5? Which of precision and recall are you choosing to raise?

#### **Class Example 11.2 - The curve, drawn**

In [ ]:
# RUN-TOGETHER
b0, b1 = -4, 0.8
xs = np.linspace(0, 10, 300)
ps = 1 / (1 + np.exp(-(b0 + b1 * xs)))

sns.lineplot(x=xs, y=ps)
plt.axhline(0.5, color='0.5', linestyle=':')
plt.axvline(-b0 / b1, color='crimson', linestyle='--')
plt.text(-b0/b1 + 0.15, 0.05, f'boundary at x = {-b0/b1:g}', color='crimson')
plt.xlabel('hours studied'); plt.ylabel('P(pass)')
plt.title('The logistic curve is bounded between 0 and 1')
plt.show()

for xi in (3, 5, 7):
    z = b0 + b1 * xi
    print(f'x = {xi}:  z = {z:+.1f}   p = {1/(1+np.exp(-z)):.4f}')

### 6.2 Logistic regression on real data

A question you already know: **who survived the Titanic?** In Homework 1 `survived` was our example of a nominal feature stored as 0 and 1. That is exactly the label logistic regression is built for.

In [ ]:
# RUN-TOGETHER
from sklearn.linear_model import LogisticRegression

titanic = sns.load_dataset('titanic')
d = titanic[['survived', 'pclass', 'age', 'fare', 'sex']].dropna()
d['is_female'] = (d['sex'] == 'female').astype(int)

features = ['pclass', 'age', 'fare', 'is_female']
Xs = StandardScaler().fit_transform(d[features])   # scale so coefficients are comparable
ys = d['survived']

Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(Xs, ys, test_size=0.25,
                                              random_state=210, stratify=ys)
log = LogisticRegression(max_iter=2000).fit(Xs_tr, ys_tr)

print('passengers with complete records:', d.shape[0])
print('test accuracy :', round(log.score(Xs_te, ys_te), 3))
print('baseline      :', round(1 - ys.mean(), 3), '(always predict died)')
print()
for name, coef in zip(features, log.coef_[0]):
    print(f'  {name:>10}: {coef:+.2f}')

Because the features were standardized, the coefficients are comparable: the largest in absolute value is the one the model leans on hardest, and a **positive** coefficient pushes toward survival.

Two dominate: `is_female` at $+1.15$, and `pclass` at $-1.10$, where a *larger* pclass number means a *lower* class of ticket. Age is weakly negative and fare barely matters once class is in the model.

> **Discuss.** The model reaches about 80% accuracy against a 59% baseline. Homework 1 asked you to notice that 177 passengers have no recorded age, and `.dropna()` has just removed every one of them. Which passengers were most likely to be missing an age, and in which direction might that bias these coefficients?

---
## 7. Cross-Validation
---
### 7.1 The problem with one split

Every score so far came from a single split with `random_state=210`. That number was arbitrary. What if it mattered?

**Definition.** **k-fold cross-validation** divides the data into $k$ equal parts. Each part takes a turn as the test set while the other $k-1$ train the model, producing $k$ scores. The **mean** estimates performance; the **spread** tells you how far to trust that estimate.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/11-supervised_regression/fig_cross_validation_training.png?raw=true" width="700">

In [ ]:
# RUN-TOGETHER
from sklearn.model_selection import cross_val_score, KFold

scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2')
print('5-fold R2, using the default:', np.round(scores, 3))
print('mean:', round(scores.mean(), 3))

**Something is badly wrong.** The last fold is $-0.72$: on those cars the model did far worse than always guessing the mean. The scores run from $-0.72$ to $0.79$ on data where a single split gave 0.70.

Before reading on, form a hypothesis. Look again at the columns of `cars`, and at the order the rows arrive in.

#### **Activity 11.4 - Diagnose the broken folds**

**Step 1.** Run the cell below, which prints the range of `model_year` in each fold.

**Step 2.** In one sentence, explain the negative $R^2$ in the final fold.

**Step 3.** Fix it by re-running with `shuffle=True` and report the new scores.

**Questions.**

**A.** The rows arrive sorted by model year, 1970 through 1982, and cars got lighter and more efficient across that period. Explain why a model trained on the first four folds predicts the last one badly, in terms of the *range of weights* it has seen.

**B.** `cross_val_score` does not shuffle by default. Give a situation where that default is right and shuffling would be a mistake.

**C.** Module 10 used `stratify=y` on the train/test split. What is the classification analogue of the problem you just found?

**D.** After shuffling, the five scores still are not identical. What does their spread tell you that the mean alone does not?

In [ ]:
# FILL-IN  (Activity 11.4)
for f, (train_idx, test_idx) in enumerate(KFold(5).split(X)):
    yr = cars.iloc[test_idx]['model_year']
    print(f'fold {f}: model years {yr.min()} to {yr.max()}')

kf = KFold(5, shuffle=____, random_state=210)
shuffled = cross_val_score(LinearRegression(), X, y, cv=kf, scoring='r2')
print()
print('shuffled 5-fold R2:', np.round(shuffled, 3))
print('mean:', round(shuffled.mean(), 3), ' spread:', round(shuffled.max() - shuffled.min(), 3))

### 7.2 Comparing models honestly

Cross-validation also fixes the problem raised at the end of Module 10, Section 2. If you try several models and report the best **test** score, the test set has quietly influenced your choice. Cross-validation lets you compare using only the training data, leaving the test set genuinely untouched for one final, honest number.

In [ ]:
# RUN-TOGETHER
from sklearn.ensemble import RandomForestRegressor
FEATURES = ['weight_1000lb', 'horsepower', 'displacement',
            'model_year', 'cylinders', 'acceleration']
kf = KFold(5, shuffle=True, random_state=210)

for name, model in [('linear regression', LinearRegression()),
                    ('tree, depth 3', DecisionTreeRegressor(max_depth=3, random_state=0)),
                    ('random forest', RandomForestRegressor(n_estimators=200, random_state=0))]:
    s = cross_val_score(model, cars[FEATURES], cars['mpg'], cv=kf, scoring='r2')
    print(f'{name:>18}:  mean R2 = {s.mean():.3f}   folds {np.round(s, 3)}')

---
## 8. What We Are Not Covering
---

The four regression models in this module are a starting kit, not the catalogue. Three families you will meet if you continue, named here so the words are not a surprise later.

**Support Vector Regression (SVR).** Instead of minimizing squared error, SVR fits a tube of a chosen width around the prediction and only penalizes points that fall *outside* it, which makes it resistant to outliers. Its real power comes from the **kernel trick**, a way of fitting curved relationships without ever explicitly constructing the curved features. That idea needs more linear algebra than this course assumes, which is why we name SVR rather than teach it: without kernels it is close to a robust version of the line you already have.

**Gradient boosting.** Like a random forest it combines many trees, but instead of averaging independent trees it grows them **in sequence**, each one trained to correct the errors the previous ones left behind. Boosted trees (XGBoost, LightGBM) win most competitions on tabular data.

**Neural networks.** Stacked layers of simple functions, fitted by gradient descent. Superb on images, audio, and text; usually *worse* than a boosted tree on a spreadsheet.

The pattern across all three is the one from Section 5: more flexibility buys accuracy and costs interpretability. Nothing in this list will tell you "7.7 mpg per 1000 pounds."

---
## Adding to the Toolkit
---

| Tool | Question it answers | Feature types it needs | Label needed? | Key assumption | How it fails |
| --- | --- | --- | --- | --- | --- |
| **Linear regression** | How much? And how much does one feature move the answer? | interval or ratio features; **numeric label** | **yes**, numeric | the relationship is a straight line | curves; outliers pull the line; impossible predictions; over-confident extrapolation |
| **k-NN regression** | How much, without assuming a shape? | interval or ratio, **scaled** | **yes**, numeric | nearby cases have similar values | `k` matters; no coefficients to report; flattens at the edges of the data |
| **Regression tree** | How much, by a readable rule? | any; **no scaling** | **yes**, numeric | a step function is close enough | overfits badly without a depth limit; unstable |
| **Random forest** | How much, as accurately as possible? | any; **no scaling** | **yes**, numeric | averaging cancels the trees' noise | no single readable rule; importances mislead when features are correlated |
| **MAE / RMSE / $R^2$** | How far off are the predictions? | numeric predictions and actuals | **yes** | one number can summarize the errors | hides structure a residual plot would show; $R^2$ can be negative |
| **Logistic regression** | What is the **probability** of a category? | interval or ratio, **scaled** for comparable coefficients | **yes**, categorical | log-odds are linear in the features | forces a linear boundary; probabilities need calibration before trusting |
| **k-fold cross-validation** | How much does my score depend on the split I happened to take? | any | **yes** | the folds represent the whole | fails silently on ordered data unless you shuffle |

**Notice the label column has split in two.** Everything in Module 10 needed a *categorical* label; the regressors here need a *numeric* one. Logistic regression sits across the line, fitted like a regression and used like a classifier, which is why its name misleads nearly everyone the first time.

## Suggested Exercises

1. A model predicting apartment rent from floor area reports $\hat{y} = 480 + 1.75x$, with $x$ in square feet, fitted on apartments between 400 and 1,600 square feet.

    a. State the slope in a full sentence, including units.

    b. Predict the rent for a 900 square foot apartment.

    c. A 3,000 square foot apartment is listed. Should you use this model? Name the problem and the limitation from Class Example 11.1 that it illustrates.

    d. What does the intercept describe, and is it meaningful here?

2. A model predicting house prices reports MAE = \$18,000 and RMSE = \$47,000 on the same test set.

    a. Explain why RMSE is so much larger than MAE.

    b. What does that gap tell you that neither number tells you alone? Which residual pattern from Section 3 would produce it?

    c. Which would you quote to a homeowner, and which to a modeller trying to improve the fit?

    d. The same model has $R^2 = -0.12$. What does a negative $R^2$ mean, and what should you do next?

3. A logistic model for loan default has $z = -2.5 + 0.04x$, where $x$ is debt-to-income ratio as a percentage.

    a. Compute the predicted probability at $x = 40$, $x = 62.5$, and $x = 85$.

    b. Find the value of $x$ at the 0.5 decision boundary.

    c. The bank decides that missing a default is five times as costly as wrongly rejecting a good applicant. Should the cutoff move above or below 0.5? Which of precision and recall are you raising?

    d. An analyst reports the coefficient as "each extra point of debt-to-income raises the default probability by 4%." Explain what is wrong with that sentence.

4. You fit three models to 200 patient records, ordered by admission date, using `cross_val_score(model, X, y, cv=5)`.

    a. Name the specific danger of the default unshuffled folds here.

    b. Describe the change you would make, and the one situation where you would *not* shuffle.

    c. Model A has mean $R^2 = 0.61$ with folds 0.58 to 0.64. Model B has mean 0.66 with folds 0.31 to 0.94. Which would you deploy, and why is the mean insufficient?

    d. A colleague reports Model B's best fold, 0.94, as the expected performance. Name the error and connect it to the rule from Module 10, Section 2.